In [1]:
"""
Take-picture mode and image-quality capture reward.

Production:
  SATELLITE.MAX_PRIMARY_CAPTURES_PER_ORBIT = 10 (arbitrary OBC memory/downlink budget)
  simulation/take_picture.py — episode capture budget from that constant
  autonomous_control/reward.py — image_quality_capture_reward via RewardConfig

Verification (easy mode — clouds off, timed overpass):
  - build_take_picture_verification_setup(): clouds=(), seed=0 fast polar pass
  - resolve_verification_capture_step(): one shutter cmd when target stripe is in FOV
  - Compare A nadir hold vs B target track at that step (expect B reward > A)

Export proof MP4s: artifacts/06-take-picture-nadir-hold.mp4, 06-take-picture-target-track.mp4
"""

'\nTake-picture mode and image-quality capture reward.\n\nProduction:\n  SATELLITE.MAX_PRIMARY_CAPTURES_PER_ORBIT = 10 (arbitrary OBC memory/downlink budget)\n  simulation/take_picture.py — episode capture budget from that constant\n  autonomous_control/reward.py — image_quality_capture_reward via RewardConfig\n\nSemantics (slice 0 — to be extended to 2 s capture window):\n  - Agent sends take-picture command at sim step k\n  - Reward uses camera_image_quality at capture frame k\n  - Reward = k_capture * quality * (1 - cloud_blocked_fraction)\n  - Budget exhausted → further commands yield zero capture reward\n\nVerification: s01_utils/take_picture_verification.py\nExport proof MP4s: artifacts/06-take-picture-nadir-hold.mp4, 06-take-picture-target-track.mp4\n'

In [2]:
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
backend_root = notebook_dir
for _ in range(6):
    if (backend_root / "simulation").is_dir():
        break
    backend_root = backend_root.parent
os.chdir(backend_root)
sys.path.insert(0, str(backend_root))
_s01_dir = backend_root / "notebooks" / "s01"
sys.path.insert(0, str(_s01_dir))
print(f"backend_root={backend_root}")

backend_root=d:\code\sem-proj-asc\backend


In [3]:
import importlib

import autonomous_control.reward as _reward
import simulation.take_picture as _tp
import s01_utils.take_picture_verification as tpv

importlib.reload(_reward)
importlib.reload(_tp)
importlib.reload(tpv)

ctx = tpv.print_take_picture_reward_gate(seed=0)
print()
tpv.print_budget_exhaustion_demo(seed=0)

d:\code\sem-proj-asc\backend\simulation\stepper.py:169: UserWarning: controller_update_interval (1 s) is not an integer multiple of simulation_timestep (0.4 s); using nearest multiple: 0.8 s (2 sim steps).
  self._controller_interval_steps, self._effective_controller_interval_s = resolve_controller_interval_steps(


Take-picture capture gate [A: OBC nadir hold]
  budget=10  k_capture=100  reward = k * quality * (1 - cloud_frac)

  cmd  cap  taken  visible  quality  cloud   reward  budget_left
    0    0  yes      no       0.2833  0.000    28.33    9
   20   20  yes      no       0.3028  0.000    30.28    8
   40   40  yes      no       0.3023  0.000    30.23    7
   60   60  yes      no       0.3014  0.000    30.14    6
   80   80  yes      no       0.3010  0.000    30.10    5
  100  100  yes      no       0.3009  0.000    30.09    4
  251  251  yes      no       0.3008  0.000    30.08    3

  captures=7  total_capture_reward=209.24  mean_quality=0.2989

Take-picture capture gate [B: OBC target track]
  budget=10  k_capture=100  reward = k * quality * (1 - cloud_frac)

  cmd  cap  taken  visible  quality  cloud   reward  budget_left
    0    0  yes      no       0.2765  1.000     0.00    9
   20   20  yes      no       0.7488  1.000     0.00    8
   40   40  yes      no       0.7966  1.000     0.0

In [4]:
import matplotlib

matplotlib.use("Agg")

ARTIFACT_DIR = backend_root / "notebooks" / "s01" / "artifacts"
tpv.export_take_picture_verification_videos(ctx, ARTIFACT_DIR)

Exporting 06-take-picture-nadir-hold.mp4 [A nadir hold]  capture cmd steps=(0, 20, 40, 60, 80, 100, 251)
[video] archived previous export -> D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\video_archive\001-06-take-picture-nadir-hold.mp4


Writing video: 100%|██████████| 113/113 [00:10<00:00, 10.95frame/s]


[mpo_video:after_export] 06-take-picture-nadir-hold.mp4 (195121 bytes, codec=h264)
[mpo_video:play] 06-take-picture-nadir-hold.mp4 (195121 bytes, codec=h264)


artifact=D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\06-take-picture-nadir-hold.mp4
Exporting 06-take-picture-target-track.mp4 [B target track]  capture cmd steps=(0, 20, 40, 60, 80, 100, 251)
[video] archived previous export -> D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\video_archive\001-06-take-picture-target-track.mp4


Writing video: 100%|██████████| 113/113 [00:09<00:00, 11.77frame/s]


[mpo_video:after_export] 06-take-picture-target-track.mp4 (217673 bytes, codec=h264)
[mpo_video:play] 06-take-picture-target-track.mp4 (217673 bytes, codec=h264)


artifact=D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\06-take-picture-target-track.mp4


{'06-take-picture-nadir-hold.mp4': WindowsPath('D:/code/sem-proj-asc/backend/notebooks/s01/artifacts/06-take-picture-nadir-hold.mp4'),
 '06-take-picture-target-track.mp4': WindowsPath('D:/code/sem-proj-asc/backend/notebooks/s01/artifacts/06-take-picture-target-track.mp4')}